In [1]:
import pandas as pd
import os
from collections import defaultdict

# Get the project root directory
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
project_root = os.path.abspath(os.path.join(notebook_dir, '../../../'))

# Define paths
cleaned_csv_path = os.path.join(project_root, 'dataset/cleaned_transcript_mapping.csv')
raw_data_path = os.path.join(project_root, 'dataset/raw')

print(f"Project Root: {project_root}")
print(f"Cleaned CSV Path: {cleaned_csv_path}")
print(f"Raw Data Path: {raw_data_path}")

# Load the cleaned data
df = pd.read_csv(cleaned_csv_path)
print(f"\nData loaded successfully!")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
print(df.head())

Project Root: d:\GitRepos\EEG-to-Text_Conformer-Transducer
Cleaned CSV Path: d:\GitRepos\EEG-to-Text_Conformer-Transducer\dataset/cleaned_transcript_mapping.csv
Raw Data Path: d:\GitRepos\EEG-to-Text_Conformer-Transducer\dataset/raw

Data loaded successfully!
Shape: (1050, 4)
Columns: ['id', 'subject', 'sentence', 'gender']

First 5 rows:
       id subject                                  sentence gender
0  1_SUB2    SUB2                          siapa wanita itu   male
1  2_SUB2    SUB2    anda benar benar kakek yang murah hati   male
2  3_SUB2    SUB2                        cobalah besok pagi   male
3  4_SUB2    SUB2   selamat sore nona ada yang bisa dibantu   male
4  5_SUB2    SUB2  untung saja ada orang baik yang menolong   male


In [2]:
# Group by subject and get unique features
subject_info = df.groupby('subject').first()[['gender']].reset_index()
print(f"Total subjects: {len(subject_info)}")
print("\nSubject Information:")
print(subject_info.to_string(index=False))

# Create a dictionary for subject -> gender mapping
subject_gender_map = dict(zip(subject_info['subject'], subject_info['gender']))

# Group IDs by subject
subject_ids = defaultdict(list)
for idx, row in df.iterrows():
    subject_ids[row['subject']].append(row['id'])

print(f"\n\nSubjects and their ID counts:")
for subject in sorted(subject_ids.keys()):
    print(f"  {subject}: {len(subject_ids[subject])} IDs")

Total subjects: 12

Subject Information:
subject gender
   SUB1   male
  SUB10 female
  SUB11 female
  SUB12 female
   SUB2   male
   SUB3   male
   SUB4   male
   SUB5   male
   SUB6 female
   SUB7 female
   SUB8   male
   SUB9 female


Subjects and their ID counts:
  SUB1: 100 IDs
  SUB10: 100 IDs
  SUB11: 100 IDs
  SUB12: 100 IDs
  SUB2: 50 IDs
  SUB3: 50 IDs
  SUB4: 100 IDs
  SUB5: 50 IDs
  SUB6: 100 IDs
  SUB7: 100 IDs
  SUB8: 100 IDs
  SUB9: 100 IDs


In [3]:
def check_csv_files_for_subject(subject, gender, subject_ids, raw_data_path):
    """
    Check if CSV files exist for a given subject
    """
    # 1. Ubah jalur ke folder 'csv'
    csv_folder = os.path.join(raw_data_path, gender, subject, 'csv')
    
    result = {
        'subject': subject,
        'gender': gender,
        'csv_folder': csv_folder,
        'folder_exists': os.path.isdir(csv_folder),
        'total_ids': len(subject_ids[subject]),
        'csv_files_found': 0,
        'missing_ids': [],
        'found_ids': []
    }
    
    # If folder doesn't exist, return early
    if not result['folder_exists']:
        result['missing_ids'] = subject_ids[subject]
        return result
    
    # List all files in the csv folder
    try:
        # 2. Filter hanya file yang berakhiran .bp.csv
        csv_files = [f for f in os.listdir(csv_folder) if f.endswith('.bp.csv')]
        result['csv_files_found'] = len(csv_files)
        
        # Check for missing IDs
        for id_val in subject_ids[subject]:
            # 3. Pengecekan ID sebagai prefix dan .bp.csv sebagai suffix
            # Tambahkan '_' setelah id_val untuk memastikan '1_DAM' tidak keliru cocok dengan '1_DAM2'
            if any(csv_file.startswith(id_val + '_') and csv_file.endswith('.bp.csv') for csv_file in csv_files):
                result['found_ids'].append(id_val)
            else:
                result['missing_ids'].append(id_val)
                
    except Exception as e:
        result['error'] = str(e)
        result['missing_ids'] = subject_ids[subject]
    
    return result

# Check all subjects
check_results = []
for subject in sorted(subject_ids.keys()):
    gender = subject_gender_map[subject]
    result = check_csv_files_for_subject(subject, gender, subject_ids, raw_data_path)
    check_results.append(result)

print(f"CSV file checking completed for {len(check_results)} subjects")

CSV file checking completed for 12 subjects


In [4]:
print("\n" + "=" * 100)
print("CSV FILE CHECK REPORT")
print("=" * 100)

total_subjects = len(check_results)
subjects_with_complete_files = 0
subjects_with_missing_files = 0
subjects_with_missing_folder = 0
total_missing_files = 0

for result in check_results:
    print(f"\n{'-' * 100}")
    print(f"SUBJECT: {result['subject']} ({result['gender']})")
    print(f"{'-' * 100}")
    
    print(f"CSV Folder: {result['csv_folder']}")
    print(f"Folder Exists: {result['folder_exists']}")
    
    if not result['folder_exists']:
        print(f"⚠️  FOLDER NOT FOUND!")
        print(f"   Missing all {result['total_ids']} IDs")
        subjects_with_missing_folder += 1
        total_missing_files += result['total_ids']
    else:
        print(f"Total IDs expected: {result['total_ids']}")
        print(f"CSV files found: {result['csv_files_found']}")
        print(f"IDs with matching CSV files: {len(result['found_ids'])}")
        print(f"IDs missing CSV files: {len(result['missing_ids'])}")
        
        if len(result['missing_ids']) == 0:
            print(f"✓ All CSV files present!")
            subjects_with_complete_files += 1
        else:
            print(f"✗ Missing CSV files for:")
            for missing_id in sorted(result['missing_ids'])[:10]:  # Show first 10
                print(f"   - {missing_id}")
            if len(result['missing_ids']) > 10:
                print(f"   ... and {len(result['missing_ids']) - 10} more")
            subjects_with_missing_files += 1
            total_missing_files += len(result['missing_ids'])

print(f"\n\n" + "=" * 100)
print("SUMMARY")
print("=" * 100)
print(f"Total subjects: {total_subjects}")
print(f"Subjects with all CSV files: {subjects_with_complete_files}")
print(f"Subjects with missing CSV files: {subjects_with_missing_files}")
print(f"Subjects with missing folder: {subjects_with_missing_folder}")
print(f"Total missing CSV files: {total_missing_files}")
print("=" * 100)


CSV FILE CHECK REPORT

----------------------------------------------------------------------------------------------------
SUBJECT: SUB1 (male)
----------------------------------------------------------------------------------------------------
CSV Folder: d:\GitRepos\EEG-to-Text_Conformer-Transducer\dataset/raw\male\SUB1\csv
Folder Exists: True
Total IDs expected: 100
CSV files found: 100
IDs with matching CSV files: 100
IDs missing CSV files: 0
✓ All CSV files present!

----------------------------------------------------------------------------------------------------
SUBJECT: SUB10 (female)
----------------------------------------------------------------------------------------------------
CSV Folder: d:\GitRepos\EEG-to-Text_Conformer-Transducer\dataset/raw\female\SUB10\csv
Folder Exists: True
Total IDs expected: 100
CSV files found: 100
IDs with matching CSV files: 100
IDs missing CSV files: 0
✓ All CSV files present!

------------------------------------------------------------

In [5]:
# Create a detailed report of missing files
missing_report = []

for result in check_results:
    if result['folder_exists'] and len(result['missing_ids']) > 0:
        for missing_id in result['missing_ids']:
            missing_report.append({
                'subject': result['subject'],
                'gender': result['gender'],
                'missing_id': missing_id,
                'csv_folder': result['csv_folder']
            })

if missing_report:
    missing_df = pd.DataFrame(missing_report)
    print(f"\nDetailed Missing Files ({len(missing_df)} total):")
    print(missing_df.to_string(index=False))
else:
    print("\nNo missing files found!")


No missing files found!


In [6]:
print("\n" + "=" * 100)
print("INVESTIGATION: SUBJECTS WITH MISSING DATA - CHECKING FOR DUPLICATE PREFIXES")
print("=" * 100)

# Filter results for subjects with missing files
subjects_with_issues = [r for r in check_results if r['folder_exists'] and len(r['missing_ids']) > 0]

if not subjects_with_issues:
    print("\nNo subjects with missing files found.")
else:
    for result in subjects_with_issues:
        subject = result['subject']
        gender = result['gender']
        csv_folder = result['csv_folder']
        missing_ids = result['missing_ids']
        
        print(f"\n{'-' * 100}")
        print(f"SUBJECT: {subject} ({gender})")
        print(f"Folder: {csv_folder}")
        print(f"Missing IDs: {len(missing_ids)}")
        print(f"{'-' * 100}")
        
        # List all valid CSV files in the folder
        try:
            all_csv_files = sorted([f for f in os.listdir(csv_folder) if f.endswith('.bp.csv')])
            print(f"\nTotal '.bp.csv' files found: {len(all_csv_files)}")
            
            # Extract prefixes from all CSV files
            prefix_to_files = defaultdict(list)
            for csv_file in all_csv_files:
                # Extract prefix (ID part: e.g., "1_SUB", "2_SUB")
                parts = csv_file.split('_')
                if len(parts) >= 2:
                    prefix = parts[0] + '_' + parts[1]  # Format: "1_SUB"
                    prefix_to_files[prefix].append(csv_file)
            
            # Check for duplicate prefixes
            duplicate_prefixes = {p: files for p, files in prefix_to_files.items() if len(files) > 1}
            
            print(f"\nUnique prefixes: {len(prefix_to_files)}")
            print(f"Prefixes with duplicates: {len(duplicate_prefixes)}")
            
            if duplicate_prefixes:
                print(f"\n⚠️  DUPLICATE PREFIXES FOUND:")
                print("-" * 100)
                for prefix in sorted(duplicate_prefixes.keys()):
                    files = duplicate_prefixes[prefix]
                    print(f"\nPrefix: {prefix}")
                    print(f"Files ({len(files)}):")
                    for f in sorted(files):
                        print(f"  - {f}")
            else:
                print(f"\n✓ No duplicate prefixes found.")
            
            # Show which expected IDs are missing
            print(f"\n\nMissing IDs Analysis:")
            print("-" * 100)
            print(f"Expected IDs but not found: {len(missing_ids)}")
            for missing_id in sorted(missing_ids)[:20]:
                # Check if there are any files with similar prefix
                similar = [p for p in prefix_to_files.keys() if missing_id.split('_')[1] in p]
                status = f"Similar prefix exists: {similar}" if similar else "No similar files"
                print(f"  {missing_id}: {status}")
            if len(missing_ids) > 20:
                print(f"  ... and {len(missing_ids) - 20} more")
                
        except Exception as e:
            print(f"\nError investigating subject {subject}: {str(e)}")

print(f"\n\n" + "=" * 100)
print("INVESTIGATION COMPLETE")
print("=" * 100)


INVESTIGATION: SUBJECTS WITH MISSING DATA - CHECKING FOR DUPLICATE PREFIXES

No subjects with missing files found.


INVESTIGATION COMPLETE
